# RSVP Experimental Dashboard

This notebook provides an integrated dashboard for experiments involving:
- Stratum Index tracking during training
- Fiber-null pruning vs. baseline
- Lamphrodynamic energy evolution during optimization
- TARTAN trajectory monitoring

Each cell is modular so you can modify or extend experiments.

In [ ]:
import torch
from torch import nn, optim
import matplotlib.pyplot as plt
import numpy as np

from rsvp_tools import (
    StratumIndexEstimator, StratumIndexConfig,
    FiberNullPruner, FiberNullConfig,
    LamFlowWrapper, LamFlowConfig,
    TARTANLayer, TARTANConfig
)

print('Dashboard loaded.')

## Define Dataset and Model for Dashboard

In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 32)
        self.fc2 = nn.Linear(32, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

def sample_data(n=256):
    x = torch.randn(n, 16)
    y = (x.sum(dim=1, keepdim=True) + 0.2 * torch.randn(n, 1))
    return x, y

model = TinyNet()
x_train, y_train = sample_data(256)
print('Dataset ready.')

## Training Loop with Stratum Index Tracking

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

si_config = StratumIndexConfig(num_samples=3, batch_size=16)
si_estimator = StratumIndexEstimator(si_config)

si_values = []
loss_values = []

for epoch in range(20):
    optimizer.zero_grad()
    pred = model(x_train)
    loss = criterion(pred, y_train)
    loss.backward()
    optimizer.step()

    # Track metrics
    si = si_estimator.estimate(
        model,
        input_sampler=lambda bs: torch.randn(bs, 16)
    )
    si_values.append(si)
    loss_values.append(loss.item())
    print(f'Epoch {epoch}: loss={loss.item():.4f}, SI={si:.2f}')

### Plot Loss and Stratum Index

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(loss_values)
plt.title('Loss over Epochs')
plt.xlabel('epoch'); plt.ylabel('loss')

plt.subplot(1,2,2)
plt.plot(si_values)
plt.title('Stratum Index over Epochs')
plt.xlabel('epoch'); plt.ylabel('SI')
plt.tight_layout(); plt.show()

## Fiber Null Pruning Comparison

In [ ]:
pruned_model = TinyNet()
pruner = FiberNullPruner(FiberNullConfig(null_rank_cutoff=1e-3))

# Before
x_eval = torch.randn(32, 16)
before = pruned_model(x_eval).detach()

# Apply projection
pruner.project_parameters(pruned_model, x_eval)

# After
after = pruned_model(x_eval).detach()

plt.plot(before[:20], label='before')
plt.plot(after[:20], label='after')
plt.legend(); plt.title('Effect of Fiber Null Projection'); plt.show()

## Lamphrodynamic Energy Tracking During Optimization

In [ ]:
lam_config = LamFlowConfig(use_hessian=False, lam_weight=0.01)
lam_model = LamFlowWrapper(TinyNet(), criterion, lam_config)
opt_lam = optim.Adam(lam_model.parameters(), lr=1e-3)

lam_energy_history = []

for epoch in range(20):
    opt_lam.zero_grad()
    loss = lam_model(x_train, y_train)
    loss.backward()
    opt_lam.step()
    lam_energy_history.append(loss.item())

plt.plot(lam_energy_history)
plt.title('Lamphrodynamic Regularized Loss Over Training')
plt.xlabel('epoch'); plt.ylabel('loss')
plt.show()

## TARTAN Dashboard: Monitoring Smoothing Over Time

In [ ]:
cfg = TARTANConfig(aura_sigma=1.0, contraction_strength=0.1, buffer_size=200)
tartan = TARTANLayer(cfg)

z = torch.randn(200, 4, 6)  # 200 timesteps, batch=4, 6 dims
z_smooth = tartan(z)[:,0,:].detach()

proj = torch.randn(6)
plt.plot((z[:,0,:]@proj).detach().numpy(), alpha=0.4, label='raw')
plt.plot((z_smooth@proj).detach().numpy(), label='smooth')
plt.title('TARTAN Temporal Smoothing'); plt.legend(); plt.show()